<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LoRA from Scratch

In [27]:
import math
import torch
import torch.nn as nn

Hyper Parameters

In [28]:
rank = 8
alpha = 2 * rank
dropout = 0.05

LoRA Linear Class

In [29]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float = dropout):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        for p in self.base.parameters():
            p.requires_grad = False

        self.A = nn.Linear(base.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base.out_features, bias=False)

        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scaling

Replace LoRA Modules

In [30]:
def replace_lora_modules(model: nn.Module, target_names: tuple, rank: int, alpha: int):
    for name, module in list(model.named_modules()):
        if any(name.endswith(t) for t in target_names) and isinstance(module, nn.Linear):
            if '.' not in name:
                parent = model
                child_name = name
            else:
                parent_name, child_name = name.rsplit(".", 1)
                parent = model.get_submodule(parent_name)
            setattr(parent, child_name, LoRALinear(module, rank, alpha))

A simple model to test

In [31]:
class SimpleModel(nn.Module):
  def __init__(self, in_features: int, out_features: int):
    super().__init__()
    self.w1 = nn.Linear(in_features, 2 * in_features)
    self.w2 = nn.Linear(2 * in_features, out_features)
    self.relu = nn.ReLU()
  def forward(self, x: torch.Tensor):
    return self.w2(self.relu(self.w1(x)))

Testing `replace_lora_modules` with `SimpleModel`

In [32]:
in_features = 10
out_features = 5
model = SimpleModel(in_features, out_features)

print("Original SimpleModel Architecture:")
print(model)

Original SimpleModel Architecture:
SimpleModel(
  (w1): Linear(in_features=10, out_features=20, bias=True)
  (w2): Linear(in_features=20, out_features=5, bias=True)
  (relu): ReLU()
)


In [33]:
target_names = ("w1", "w2")

# Apply LoRA to the model
replace_lora_modules(model, target_names, rank, alpha)

print("\nSimpleModel Architecture after applying LoRA:")
print(model)


SimpleModel Architecture after applying LoRA:
SimpleModel(
  (w1): LoRALinear(
    (base): Linear(in_features=10, out_features=20, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
    (A): Linear(in_features=10, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=20, bias=False)
  )
  (w2): LoRALinear(
    (base): Linear(in_features=20, out_features=5, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
    (A): Linear(in_features=20, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=5, bias=False)
  )
  (relu): ReLU()
)


In [34]:
# Verify that the layers have been replaced with LoRALinear
assert isinstance(model.w1, LoRALinear), "w1 was not replaced by LoRALinear"
assert isinstance(model.w2, LoRALinear), "w2 was not replaced by LoRALinear"
print("\nVerification successful: w1 and w2 are now LoRALinear instances.")


Verification successful: w1 and w2 are now LoRALinear instances.
